# EndoGSLAM with Innovations - 10-Scene Evaluation

**Best config: BA + Visibility Pruning (eta=0.90)**

Runs all 10 C3VD scenes sequentially with reduced logging to prevent browser freezing.

Runtime: ~60-90 min per scene on L4 GPU (~10-15 hours total)

## 1. Environment Setup

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Clone repository
!git clone https://github.com/baimingyang98/Endoscopic-3DGS-SLAM-with-Visibility-Pruning.git /content/project
%cd /content/project

In [ ]:
# Install dependencies
!pip install -q tqdm numpy Pillow opencv-python imageio matplotlib kornia natsort pyyaml plotly lpips open3d torchmetrics pytorch-msssim trimesh

## 2. Install Modified CUDA Rasterizer

In [ ]:
# Clone base rasterizer
!git clone https://github.com/JonathonLuiten/diff-gaussian-rasterization-w-depth.git /content/rasterizer

In [ ]:
# Apply visibility patches (uses the patch script from our repo)
!python patch_rasterizer.py /content/rasterizer

In [ ]:
# Build and install
!pip install /content/rasterizer/

In [ ]:
# Verify installation
import torch
from diff_gaussian_rasterization import GaussianRasterizer, GaussianRasterizationSettings
print(f'CUDA: {torch.cuda.get_device_name(0)}')
print('Rasterizer import OK')

## 3. Download C3VD Dataset

In [ ]:
!pip install -q gdown
!mkdir -p data/C3VD
!gdown 1MwpfFKKweM1L3bYY7V4dYzS_IlkFaOUe -O /content/C3VD_EndoGSLAM.tar.gz
!tar -xzf /content/C3VD_EndoGSLAM.tar.gz -C data/
!ls data/C3VD/

## 4. Write Optimized Config (eta=0.90, reduced logging)

In [ ]:
%%writefile configs/c3vd/c3vd_best.py
"""
Best configuration: BA + Visibility Pruning (eta=0.90)
Optimized for batch 10-scene runs with minimal logging.
"""
import os

scenes = [
    "cecum_t1_b",
    "cecum_t2_b",
    "cecum_t3_a",
    "sigmoid_t1_a",
    "sigmoid_t2_a",
    "sigmoid_t3_a",
    "trans_t1_b",
    "trans_t2_c",
    "trans_t4_a",
    "trans_t4_b",
]

primary_device = "cuda:0"
seed = 0

try:
    scene_name = scenes[int(os.environ["SCENE_NUM"])]
except (KeyError, IndexError):
    scene_name = "sigmoid_t3_a"

map_every = 1
keyframe_every = 8
tracking_iters = 30
mapping_iters = 50

group_name = "C3VD_best"
run_name = scene_name

config = dict(
    workdir=f"./experiments/{group_name}",
    run_name=run_name,
    seed=seed,
    primary_device=primary_device,
    map_every=map_every,
    keyframe_every=keyframe_every,
    distance_keyframe_selection=True,
    distance_current_frame_prob=0.1,
    mapping_window_size=-1,
    report_global_progress_every=999999,
    scene_radius_depth_ratio=3,
    mean_sq_dist_method="projective",
    report_iter_progress=False,
    load_checkpoint=False,
    checkpoint_time_idx=0,
    save_checkpoints=False,
    checkpoint_interval=int(1e10),
    gaussian_simplification=True,
    data=dict(
        basedir="./data/C3VD",
        gradslam_data_cfg="./configs/data/c3vd.yaml",
        sequence=scene_name,
        desired_image_height=1080 // 2,
        desired_image_width=1350 // 2,
        start=0,
        end=-1,
        stride=1,
        num_frames=-1,
        train_or_test="train",
    ),
    tracking=dict(
        use_gt_poses=False,
        forward_prop=True,
        num_iters=tracking_iters,
        use_sil_for_loss=True,
        sil_thres=0.99,
        use_l1=True,
        ignore_outlier_depth_loss=False,
        loss_weights=dict(im=0.5, depth=1.0),
        lrs=dict(
            means3D=0.0, rgb_colors=0.0, unnorm_rotations=0.0,
            logit_opacities=0.0, log_scales=0.0,
            cam_unnorm_rots=0.002, cam_trans=0.005,
        ),
    ),
    mapping=dict(
        num_iters=mapping_iters,
        add_new_gaussians=True,
        sil_thres=0.5,
        use_l1=True,
        use_sil_for_loss=False,
        ignore_outlier_depth_loss=False,
        loss_weights=dict(im=1.0, depth=1.0),
        lrs=dict(
            means3D=0.0001, rgb_colors=0.0025, unnorm_rotations=0.001,
            logit_opacities=0.05, log_scales=0.001,
            cam_unnorm_rots=0.000, cam_trans=0.000,
        ),
        prune_gaussians=True,
        pruning_dict=dict(
            start_after=0, remove_big_after=0, stop_after=20,
            prune_every=20, removal_opacity_threshold=0.005,
            final_removal_opacity_threshold=0.005,
            reset_opacities=False, reset_opacities_every=int(1e10),
        ),
        use_gaussian_splatting_densification=False,
        densify_dict=dict(
            start_after=500, remove_big_after=3000, stop_after=5000,
            densify_every=100, grad_thresh=0.0002, num_to_split_into=2,
            removal_opacity_threshold=0.005, final_removal_opacity_threshold=0.005,
            reset_opacities_every=3000,
        ),
    ),
    innovations=dict(
        enable_visibility_pruning=True,
        distance_gamma=0.5,
        degeneration_eta=0.90,
        vis_threshold=0.05,
        min_observations=50,
        vis_window_size=15,
        enable_periodic_ba=True,
        ba_every_m_frames=50,
        ba_n_keyframes=5,
        ba_num_iters=20,
        ba_selection="hybrid",
        ba_lrs=dict(
            means3D=0.00005, rgb_colors=0.001, unnorm_rotations=0.0005,
            logit_opacities=0.025, log_scales=0.0005,
            cam_unnorm_rots=0.001, cam_trans=0.002,
        ),
        enable_deformation=False,
        deform_lr=0.0005,
        var_threshold=0.1,
        lambda_deform_temporal=0.1,
        lambda_deform_magnitude=0.01,
        enable_deform_weighted_tracking=False,
    ),
    viz=dict(
        render_mode="color",
        offset_first_viz_cam=True,
        show_sil=False,
        visualize_cams=False,
        viz_w=320, viz_h=320,
        viz_near=0.01, viz_far=100.0,
        view_scale=2, viz_fps=30,
        enter_interactive_post_online=True,
    ),
)

## 5. Run All 10 Scenes

In [ ]:
import subprocess
import time
import os

scenes = [
    "cecum_t1_b",    # 0
    "cecum_t2_b",    # 1
    "cecum_t3_a",    # 2
    "sigmoid_t1_a",  # 3
    "sigmoid_t2_a",  # 4
    "sigmoid_t3_a",  # 5
    "trans_t1_b",    # 6
    "trans_t2_c",    # 7
    "trans_t4_a",    # 8
    "trans_t4_b",    # 9
]

results = {}

for idx, scene in enumerate(scenes):
    print(f"\n{'='*60}")
    print(f"Scene {idx+1}/10: {scene}")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    env = os.environ.copy()
    env['SCENE_NUM'] = str(idx)
    
    # Run SLAM - redirect verbose output to log file
    log_file = f'experiments/C3VD_best/{scene}/run.log'
    os.makedirs(f'experiments/C3VD_best/{scene}', exist_ok=True)
    
    with open(log_file, 'w') as log:
        proc = subprocess.run(
            ['python', 'scripts/main.py', 'configs/c3vd/c3vd_best.py'],
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            cwd='/content/project'
        )
    
    elapsed = time.time() - start_time
    status = 'OK' if proc.returncode == 0 else f'FAILED (code {proc.returncode})'
    results[scene] = {'status': status, 'time_min': elapsed / 60}
    
    print(f"  Status: {status}")
    print(f"  Time: {elapsed/60:.1f} min")
    
    if proc.returncode != 0:
        # Print last 20 lines of log on failure
        with open(log_file, 'r') as f:
            lines = f.readlines()
        print("  Last 20 lines of log:")
        for line in lines[-20:]:
            print(f"    {line.rstrip()}")

print(f"\n{'='*60}")
print("ALL SCENES COMPLETE")
print(f"{'='*60}")
for scene, info in results.items():
    print(f"  {scene:20s} | {info['status']:10s} | {info['time_min']:.1f} min")

## 6. Compute Metrics

In [ ]:
!python scripts/calc_metrics.py --all --group_dir ./experiments/C3VD_best --data_dir ./data/C3VD

In [ ]:
# Display results table
import pandas as pd
import os

csv_path = 'experiments/C3VD_best/metrics_summary.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(df.to_string(index=False))
    print(f"\n--- Averages ---")
    for col in df.columns:
        if col != 'scene':
            print(f"  {col}: {df[col].mean():.4f}")
else:
    print('No metrics CSV found. Check if scenes completed successfully.')

## 7. Download Results

In [ ]:
# Package results for download
!tar -czf /content/results_C3VD_best.tar.gz \
    experiments/C3VD_best/metrics_summary.csv \
    experiments/C3VD_best/*/eval/est_w2c.txt \
    experiments/C3VD_best/*/eval/gt_train_w2c.txt \
    experiments/C3VD_best/*/runtimes.txt

print('Results packaged: /content/results_C3VD_best.tar.gz')